# Data Contract: Ranking Signal Analysis

## 1. Unit of Analysis
One row = one **content item (page)** per **month**.

## 2. Time Window
We will use a mid-panel month partition (e.g., `month=2026-03`) for EDA and feature engineering. The final month (June 2026) will be strictly reserved as the sealed test set.

## 3. Field Classification
- **Features (Knowable before outcome):** word_count, has_video, content_type, etc.
- **Label (Outcome to predict):** avg_position, ctr, engagement_rate (in the subsequent month or current month depending on formulation).
- **Context (For grouping/joining):** content_id, client_id, report_date.
- **Excluded:** is_declining_label, trend_direction, trend_pct (these are derived and cause leakage if used as features).

## 4. Missing Values
We expect missingness to follow patterns (e.g., keyword data missing for certain content types). We will investigate these using grouped queries.

## 5. Output
A ranked list of content signals (e.g., word count > X, specific formats) ordered by their correlation with high visibility (low avg_position) and engagement.

In [ ]:
# Import libraries
import duckdb
import pandas as pd
import os

# Configure DuckDB to use Hugging Face token
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    print('Please set HF_TOKEN environment variable')

conn = duckdb.connect(':memory:')
conn.execute(f"""
    INSTALL httpfs;
    LOAD httpfs;
    SET bearer_token='{hf_token}';
""")

In [ ]:
# Verify Grain: One row per content item in dim_content
conn.execute("""
    SELECT content_id, COUNT(*) as c 
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content/*.parquet'
    GROUP BY content_id 
    HAVING c > 1 
    LIMIT 5
""").df()